In [1]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

# Load dataset
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Flatten and normalize
x_train = x_train.reshape(x_train.shape[0], 784) / 255.0
x_test = x_test.reshape(x_test.shape[0], 784) / 255.0

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
def one_hot_encode(y, num_classes=10):

    one_hot = np.zeros((y.size, num_classes))

    one_hot[np.arange(y.size), y] = 1

    return one_hot

y_train_encoded = one_hot_encode(y_train)
y_test_encoded = one_hot_encode(y_test)

In [3]:
def relu(z):
    return np.maximum(0, z)

In [4]:
def relu_derivative(z):
    return (z > 0).astype(float)

In [5]:
def softmax(z):

    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))

    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [7]:
def cross_entropy_loss(y_true, y_pred):

    epsilon = 1e-10

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

    return loss

In [14]:
import numpy as np


# ReLU Activation
def relu(z):
    return np.maximum(0, z)


# ReLU Derivative
def relu_derivative(z):
    return (z > 0).astype(float)


# Softmax Function
def softmax(z):

    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))

    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# Feedforward Neural Network Class
class FeedForwardNeuralNetwork:

    def __init__(self, input_size, hidden_layers, output_size):

        # Define network architecture
        self.layers = [input_size] + hidden_layers + [output_size]

        # Store weights and biases
        self.weights = []
        self.biases = []

        # Xavier Weight Initialization
        for i in range(len(self.layers) - 1):

            weight = np.random.randn(
                self.layers[i],
                self.layers[i + 1]
            ) * np.sqrt(1 / self.layers[i])

            bias = np.zeros((1, self.layers[i + 1]))

            self.weights.append(weight)
            self.biases.append(bias)

    # Forward Propagation
    def forward(self, X):

        activations = [X]
        z_values = []

        A = X

        # Hidden Layers
        for i in range(len(self.weights) - 1):

            Z = np.dot(A, self.weights[i]) + self.biases[i]

            z_values.append(Z)

            A = relu(Z)

            activations.append(A)

        # Output Layer
        Z = np.dot(A, self.weights[-1]) + self.biases[-1]

        z_values.append(Z)

        output = softmax(Z)

        activations.append(output)

        return activations, z_values

    # Backpropagation
    def backward(self, X, y, activations, z_values):

        m = X.shape[0]

        gradients_w = []
        gradients_b = []

        # Output layer error
        dZ = activations[-1] - y

        # Backpropagation loop
        for i in reversed(range(len(self.weights))):

            dW = np.dot(activations[i].T, dZ) / m

            dB = np.sum(dZ, axis=0, keepdims=True) / m

            gradients_w.insert(0, dW)
            gradients_b.insert(0, dB)

            # Hidden layer gradients
            if i > 0:

                dA = np.dot(dZ, self.weights[i].T)

                dZ = dA * relu_derivative(z_values[i - 1])

        return gradients_w, gradients_b

In [11]:
    def update_parameters_sgd(self, gradients_w, gradients_b, learning_rate):

        for i in range(len(self.weights)):

            self.weights[i] -= learning_rate * gradients_w[i]

            self.biases[i] -= learning_rate * gradients_b[i]

In [22]:
        def __init__():
          self.v_w = [np.zeros_like(w) for w in self.weights]
          self.v_b = [np.zeros_like(b) for b in self.biases]
          self.s_w = [np.zeros_like(w) for w in self.weights]
          self.s_b = [np.zeros_like(b) for b in self.biases]
          self.m_w = [np.zeros_like(w) for w in self.weights]
          self.m_b = [np.zeros_like(b) for b in self.biases]

          self.v_w_adam = [np.zeros_like(w) for w in self.weights]
          self.v_b_adam = [np.zeros_like(b) for b in self.biases]

In [20]:
    def update_parameters_momentum(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        beta=0.9
    ):

        for i in range(len(self.weights)):

            self.v_w[i] = beta * self.v_w[i] + (1 - beta) * gradients_w[i]

            self.v_b[i] = beta * self.v_b[i] + (1 - beta) * gradients_b[i]

            self.weights[i] -= learning_rate * self.v_w[i]

            self.biases[i] -= learning_rate * self.v_b[i]

In [21]:
    def update_parameters_rmsprop(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        beta=0.9,
        epsilon=1e-8
    ):

        for i in range(len(self.weights)):

            self.s_w[i] = beta * self.s_w[i] + (1 - beta) * (gradients_w[i] ** 2)

            self.s_b[i] = beta * self.s_b[i] + (1 - beta) * (gradients_b[i] ** 2)

            self.weights[i] -= (
                learning_rate *
                gradients_w[i] /
                (np.sqrt(self.s_w[i]) + epsilon)
            )

            self.biases[i] -= (
                learning_rate *
                gradients_b[i] /
                (np.sqrt(self.s_b[i]) + epsilon)
            )

In [23]:
    def update_parameters_adam(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        t,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-8
    ):

        for i in range(len(self.weights)):

            # Momentum

            self.m_w[i] = beta1 * self.m_w[i] + (1 - beta1) * gradients_w[i]

            self.m_b[i] = beta1 * self.m_b[i] + (1 - beta1) * gradients_b[i]

            # RMSProp

            self.v_w_adam[i] = beta2 * self.v_w_adam[i] + (1 - beta2) * (gradients_w[i] ** 2)

            self.v_b_adam[i] = beta2 * self.v_b_adam[i] + (1 - beta2) * (gradients_b[i] ** 2)

            # Bias correction

            m_w_hat = self.m_w[i] / (1 - beta1 ** t)

            m_b_hat = self.m_b[i] / (1 - beta1 ** t)

            v_w_hat = self.v_w_adam[i] / (1 - beta2 ** t)

            v_b_hat = self.v_b_adam[i] / (1 - beta2 ** t)

            # Update

            self.weights[i] -= (
                learning_rate *
                m_w_hat /
                (np.sqrt(v_w_hat) + epsilon)
            )

            self.biases[i] -= (
                learning_rate *
                m_b_hat /
                (np.sqrt(v_b_hat) + epsilon)
            )

In [27]:
# Create Model

model = FeedForwardNeuralNetwork(
    input_size=784,
    hidden_layers=[128, 64],
    output_size=10
)

# Training Settings

epochs = 5
learning_rate = 0.001

# Training Loop

for epoch in range(epochs):

    # Forward Pass
    activations, z_values = model.forward(x_train[:1000])

    # Compute Loss
    loss = cross_entropy_loss(
        y_train_encoded[:1000],
        activations[-1]
    )

    # Backward Pass
    gradients_w, gradients_b = model.backward(
        x_train[:1000],
        y_train_encoded[:1000],
        activations,
        z_values
    )

    # Update Parameters using Adam
    model.update_parameters(
        gradients_w,
        gradients_b,
        learning_rate,
        optimizer="adam",
        t=epoch + 1
    )

    # Print Loss
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

Epoch 1, Loss: 2.3114
Epoch 2, Loss: 2.1549
Epoch 3, Loss: 2.0342
Epoch 4, Loss: 1.9256
Epoch 5, Loss: 1.8142


In [26]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist


# =========================
# LOAD DATASET
# =========================

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()


# =========================
# PREPROCESS DATA
# =========================

# Flatten 28x28 images into 784 vectors
x_train = x_train.reshape(x_train.shape[0], 784) / 255.0
x_test = x_test.reshape(x_test.shape[0], 784) / 255.0


# =========================
# ONE HOT ENCODING
# =========================

def one_hot_encode(y, num_classes=10):

    one_hot = np.zeros((y.size, num_classes))

    one_hot[np.arange(y.size), y] = 1

    return one_hot


y_train_encoded = one_hot_encode(y_train)
y_test_encoded = one_hot_encode(y_test)


# =========================
# ACTIVATION FUNCTIONS
# =========================

def relu(z):

    return np.maximum(0, z)


def relu_derivative(z):

    return (z > 0).astype(float)


def softmax(z):

    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))

    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# =========================
# LOSS FUNCTION
# =========================

def cross_entropy_loss(y_true, y_pred):

    epsilon = 1e-10

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

    return loss


# =========================
# NEURAL NETWORK CLASS
# =========================

class FeedForwardNeuralNetwork:

    def __init__(self, input_size, hidden_layers, output_size):

        # Network structure
        self.layers = [input_size] + hidden_layers + [output_size]

        # Store weights and biases
        self.weights = []
        self.biases = []

        # Xavier Initialization
        for i in range(len(self.layers) - 1):

            weight = np.random.randn(
                self.layers[i],
                self.layers[i + 1]
            ) * np.sqrt(1 / self.layers[i])

            bias = np.zeros((1, self.layers[i + 1]))

            self.weights.append(weight)
            self.biases.append(bias)

        # Momentum variables
        self.v_w = [np.zeros_like(w) for w in self.weights]
        self.v_b = [np.zeros_like(b) for b in self.biases]

        # RMSProp variables
        self.s_w = [np.zeros_like(w) for w in self.weights]
        self.s_b = [np.zeros_like(b) for b in self.biases]

        # Adam variables
        self.m_w = [np.zeros_like(w) for w in self.weights]
        self.m_b = [np.zeros_like(b) for b in self.biases]

        self.v_w_adam = [np.zeros_like(w) for w in self.weights]
        self.v_b_adam = [np.zeros_like(b) for b in self.biases]

    # =========================
    # FORWARD PROPAGATION
    # =========================

    def forward(self, X):

        activations = [X]
        z_values = []

        A = X

        # Hidden layers
        for i in range(len(self.weights) - 1):

            Z = np.dot(A, self.weights[i]) + self.biases[i]

            z_values.append(Z)

            A = relu(Z)

            activations.append(A)

        # Output layer
        Z = np.dot(A, self.weights[-1]) + self.biases[-1]

        z_values.append(Z)

        output = softmax(Z)

        activations.append(output)

        return activations, z_values

    # =========================
    # BACKPROPAGATION
    # =========================

    def backward(self, X, y, activations, z_values):

        m = X.shape[0]

        gradients_w = []
        gradients_b = []

        # Output layer gradient
        dZ = activations[-1] - y

        # Backpropagation loop
        for i in reversed(range(len(self.weights))):

            dW = np.dot(activations[i].T, dZ) / m

            dB = np.sum(dZ, axis=0, keepdims=True) / m

            gradients_w.insert(0, dW)
            gradients_b.insert(0, dB)

            # Hidden layer gradients
            if i > 0:

                dA = np.dot(dZ, self.weights[i].T)

                dZ = dA * relu_derivative(z_values[i - 1])

        return gradients_w, gradients_b

    # =========================
    # SGD OPTIMIZER
    # =========================

    def sgd(self, gradients_w, gradients_b, learning_rate):

        for i in range(len(self.weights)):

            self.weights[i] -= learning_rate * gradients_w[i]

            self.biases[i] -= learning_rate * gradients_b[i]

    # =========================
    # MOMENTUM OPTIMIZER
    # =========================

    def momentum(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        beta=0.9
    ):

        for i in range(len(self.weights)):

            self.v_w[i] = (
                beta * self.v_w[i]
                + (1 - beta) * gradients_w[i]
            )

            self.v_b[i] = (
                beta * self.v_b[i]
                + (1 - beta) * gradients_b[i]
            )

            self.weights[i] -= learning_rate * self.v_w[i]

            self.biases[i] -= learning_rate * self.v_b[i]

    # =========================
    # RMSPROP OPTIMIZER
    # =========================

    def rmsprop(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        beta=0.9,
        epsilon=1e-8
    ):

        for i in range(len(self.weights)):

            self.s_w[i] = (
                beta * self.s_w[i]
                + (1 - beta) * (gradients_w[i] ** 2)
            )

            self.s_b[i] = (
                beta * self.s_b[i]
                + (1 - beta) * (gradients_b[i] ** 2)
            )

            self.weights[i] -= (
                learning_rate *
                gradients_w[i] /
                (np.sqrt(self.s_w[i]) + epsilon)
            )

            self.biases[i] -= (
                learning_rate *
                gradients_b[i] /
                (np.sqrt(self.s_b[i]) + epsilon)
            )

    # =========================
    # ADAM OPTIMIZER
    # =========================

    def adam(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        t,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-8
    ):

        for i in range(len(self.weights)):

            # Momentum
            self.m_w[i] = (
                beta1 * self.m_w[i]
                + (1 - beta1) * gradients_w[i]
            )

            self.m_b[i] = (
                beta1 * self.m_b[i]
                + (1 - beta1) * gradients_b[i]
            )

            # RMSProp
            self.v_w_adam[i] = (
                beta2 * self.v_w_adam[i]
                + (1 - beta2) * (gradients_w[i] ** 2)
            )

            self.v_b_adam[i] = (
                beta2 * self.v_b_adam[i]
                + (1 - beta2) * (gradients_b[i] ** 2)
            )

            # Bias correction
            m_w_hat = self.m_w[i] / (1 - beta1 ** t)

            m_b_hat = self.m_b[i] / (1 - beta1 ** t)

            v_w_hat = self.v_w_adam[i] / (1 - beta2 ** t)

            v_b_hat = self.v_b_adam[i] / (1 - beta2 ** t)

            # Parameter update
            self.weights[i] -= (
                learning_rate *
                m_w_hat /
                (np.sqrt(v_w_hat) + epsilon)
            )

            self.biases[i] -= (
                learning_rate *
                m_b_hat /
                (np.sqrt(v_b_hat) + epsilon)
            )

    # =========================
    # FLEXIBLE OPTIMIZER CALL
    # =========================

    def update_parameters(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        optimizer="sgd",
        t=1
    ):

        if optimizer == "sgd":

            self.sgd(
                gradients_w,
                gradients_b,
                learning_rate
            )

        elif optimizer == "momentum":

            self.momentum(
                gradients_w,
                gradients_b,
                learning_rate
            )

        elif optimizer == "rmsprop":

            self.rmsprop(
                gradients_w,
                gradients_b,
                learning_rate
            )

        elif optimizer == "adam":

            self.adam(
                gradients_w,
                gradients_b,
                learning_rate,
                t
            )

        else:

            raise ValueError("Invalid optimizer selected")


# =========================
# CREATE MODEL
# =========================

model = FeedForwardNeuralNetwork(
    input_size=784,
    hidden_layers=[128, 64],
    output_size=10
)


# =========================
# TRAINING LOOP
# =========================

epochs = 5
learning_rate = 0.001

for epoch in range(epochs):

    # Forward propagation
    activations, z_values = model.forward(x_train[:1000])

    # Compute loss
    loss = cross_entropy_loss(
        y_train_encoded[:1000],
        activations[-1]
    )

    # Backpropagation
    gradients_w, gradients_b = model.backward(
        x_train[:1000],
        y_train_encoded[:1000],
        activations,
        z_values
    )

    # Update parameters
    model.update_parameters(
        gradients_w,
        gradients_b,
        learning_rate,
        optimizer="adam",
        t=epoch + 1
    )

    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

Epoch 1, Loss: 2.3169
Epoch 2, Loss: 2.1878
Epoch 3, Loss: 2.0912
Epoch 4, Loss: 1.9980
Epoch 5, Loss: 1.8997


# Task 3: Backpropagation and Optimization Algorithms

## Objective
The objective of this task is to implement the backpropagation algorithm from scratch using NumPy and train the feedforward neural network on the Fashion-MNIST dataset using different optimization algorithms.

The implementation is designed to be modular and flexible, allowing easy integration of different optimizers and support for varying batch sizes and network architectures.

---

# Dataset
The Fashion-MNIST dataset is used for training and evaluation.

Dataset details:
- 60,000 training images
- 10,000 testing images
- 10 output classes
- Image size: 28 × 28 pixels
- Flattened input dimension: 784

---

# Data Preprocessing

The following preprocessing steps are applied:

1. Flattening:
   - Images are converted from 28×28 matrices into 784-dimensional vectors.

2. Normalization:
   - Pixel values are scaled from [0,255] to [0,1].

3. One-Hot Encoding:
   - Class labels are converted into one-hot encoded vectors for multi-class classification.

Example:
```text
3 → [0 0 0 1 0 0 0 0 0 0]